# Fast Agent (Cloudflare Workers AI) — Build a SQL Agent fast!

Same flow as `L1_fast_agent.ipynb`, but the chat model is **Cloudflare Workers AI** via `ChatCloudflareWorkersAI`. Compared to the OpenAI path:

- **Credentials:** set `CF_AI_API_KEY` and `CF_ACCOUNT_ID` in `.env` (see `example.env`).
- **Models:** Workers AI uses binding names like `@cf/meta/llama-3.3-70b-instruct-fp8-fast`; override with `CF_WORKERS_AI_MODEL` if needed.
- **Code:** use `ChatCloudflareWorkersAI` explicitly; Cloudflare is not exposed as a `provider:model` string on `init_chat_model` in current LangChain docs.

<img src="./assets/LC_L1_top.png" align="left" width="500">

## Setup

Load and/or check environment variables. For this notebook, set `CF_AI_API_KEY` and `CF_ACCOUNT_ID`. Optional: `CF_WORKERS_AI_MODEL`. Other lessons may use `OPENAI_API_KEY`; keys in `example.env` are checked here.

In [1]:
from dotenv import load_dotenv
from env_utils import doublecheck_env, doublecheck_pkgs

# Load environment variables from .env
load_dotenv()

# Check and print results
doublecheck_env("example.env")  # check environmental variables
doublecheck_pkgs(pyproject_path="pyproject.toml", verbose=True)   # check packages

OPENAI_API_KEY=****here
LANGSMITH_API_KEY=****here
LANGSMITH_TRACING=true
LANGSMITH_PROJECT=****ials
GROQ_API_KEY=<not set>
CF_AI_API_KEY=<not set>
CF_ACCOUNT_ID=<not set>
Python 3.11.11 satisfies requires-python: >=3.11,<3.14
package                | required | installed | status              | path                                                                
---------------------- | -------- | --------- | ------------------- | --------------------------------------------------------------------
langgraph              | >=1.0.0  | 1.0.10rc1 | ⚠️ Version mismatch | C:\MarkDev\lca-langchainV1-essentials\python\.venv\Lib\site-packages
langchain              | >=1.0.0  | 1.2.6     | ✅ OK                | C:\MarkDev\lca-langchainV1-essentials\python\.venv\Lib\site-packages
langchain-core         | >=1.0.0  | 1.2.22    | ✅ OK                | C:\MarkDev\lca-langchainV1-essentials\python\.venv\Lib\site-packages
langchain-openai       | >=1.0.0  | 1.1.7     | ✅ OK                | C:\MarkD

In [ ]:
from langchain_community.utilities import SQLDatabase

db = SQLDatabase.from_uri("sqlite:///Chinook.db")

Define the runtime context to provide the agent and tools with access to the database.

In [ ]:
from dataclasses import dataclass

from langchain_community.utilities import SQLDatabase


# define context structure to support dependency injection
@dataclass
class RuntimeContext:
    db: SQLDatabase

<b>⚠️ Security Note:</b> This demo does not include a filter on LLM-generated commands. In production, you would want to limit the scope of LLM-generated commands. ⚠️   
This tool will connect to the database. Note the use of `get_runtime` to access the graph **runtime context**.

In [ ]:
from langchain_core.tools import tool
from langgraph.runtime import get_runtime

@tool
def execute_sql(query: str) -> str:
    """Execute a SQLite command and return results."""
    runtime = get_runtime(RuntimeContext)
    db = runtime.context.db

    try:
        return db.run(query)
    except Exception as e:
        return f"Error: {e}"

Add a system prompt to define your agents behavior.

In [ ]:
SYSTEM_PROMPT = """You are a careful SQLite analyst.

Rules:
- Think step-by-step.
- When you need data, call the tool `execute_sql` with ONE SELECT query.
- Read-only only; no INSERT/UPDATE/DELETE/ALTER/DROP/CREATE/REPLACE/TRUNCATE.
- Limit to 5 rows of output unless the user explicitly asks otherwise.
- If the tool returns 'Error:', revise the SQL and try again.
- Prefer explicit column lists; avoid SELECT *.
"""

Create your agent with **Cloudflare Workers AI** (`langchain-cloudflare`), tools, prompt, and runtime context. See [ChatCloudflareWorkersAI](https://docs.langchain.com/oss/python/integrations/chat/cloudflare_workersai) and [integrations](https://docs.langchain.com/oss/python/integrations/providers).

In [ ]:
import os

from langchain.agents import create_agent
from langchain_cloudflare.chat_models import ChatCloudflareWorkersAI

model = ChatCloudflareWorkersAI(
    model=os.getenv("CF_WORKERS_AI_MODEL", "@cf/meta/llama-3.3-70b-instruct-fp8-fast"),
    temperature=0,
    max_tokens=1024,
)

agent = create_agent(
    model=model,
    tools=[execute_sql],
    system_prompt=SYSTEM_PROMPT,
    context_schema=RuntimeContext,
)

Here's a display of the agent ReAct Loop.

In [ ]:
from IPython.display import Image, display

display(Image(agent.get_graph(xray=True).draw_mermaid_png()))

**Streaming note:** The Cloudflare Workers AI LangChain integration may not support the same **token-level** streaming as some other providers; `agent.stream` with `stream_mode="values"` still advances the graph step-by-step so you see each message.

Run some queries. Notice:
- The agent does not have the database schema and will need to discover it independently.
- The agent may make mistakes! By returning error messages, the agent can self-correct its queries.
- Notice you invoke the agent with `agent.stream`.
    - This command and the `pretty_print` display the **messages** that communicate information between the model and the tools.
- Notice the agent doesn't remember the schema between invocations... More on this later!

In [ ]:
question = "Which table has the largest number of entries?"

for step in agent.stream(
    {"messages": question},
    context=RuntimeContext(db=db),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

In [ ]:
question = "Which genre on average has the longest tracks?"

for step in agent.stream(
    {"messages": question},
    context=RuntimeContext(db=db),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

In [ ]:
question = "Please list all of the tables"

for step in agent.stream(
    {"messages": question},
    context=RuntimeContext(db=db),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

**Create your own query here!**  Add some questions of your own.

In [ ]:
question = "TRY YOUR OWN QUERY HERE"

for step in agent.stream(
    {"messages": question},
    context=RuntimeContext(db=db),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

### Let's try this Studio